# 🌿 YOLO26s — Grass Plant Detection Training
**Model:** YOLO26s (Ultralytics latest — 2025)  
**mAP@50-95:** 48.6 | **Params:** 9.5M | **Speed T4:** 2.5ms  
**Task:** Grass plant detection  
**IDE:** Kaggle (GPU T4 x2)


## 1. Install Latest Ultralytics (must be ≥ v8.4.23 for YOLO26)

In [ ]:
# Install latest ultralytics to get YOLO26 support
!pip install -q --upgrade ultralytics

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")
ultralytics.checks()

## 2. Imports

In [ ]:
import os
import shutil
import random
import yaml
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from ultralytics import YOLO
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
print("✅ Imports done")

## 3. Setup Dataset Paths

Upload your `grass_annotations.zip` as a Kaggle Dataset first,  
then set the dataset name below.

In [ ]:
# ── CHANGE THIS to your uploaded dataset name on Kaggle ──────────────────────
ANNOTATION_ZIP_DIR = Path("/kaggle/input/grass-annotations")  # ← your dataset folder name

# Source paths (from your downloaded annotation zip)
SRC_IMAGES = ANNOTATION_ZIP_DIR / "annotated_images"    # 10 annotated .jpg files
SRC_LABELS = ANNOTATION_ZIP_DIR / "labels_yolo"         # 10 .txt YOLO label files

# Verify
images = sorted(SRC_IMAGES.glob("*.jpg")) + sorted(SRC_IMAGES.glob("*.png"))
labels = sorted(SRC_LABELS.glob("*.txt"))

print(f"Images found : {len(images)}")
print(f"Labels found : {len(labels)}")
print()
for img in images:
    print(f"  {img.name}")

## 4. Build Train / Val Split (80 / 20)

In [ ]:
DATASET_DIR = Path("/kaggle/working/grass_dataset")

for split in ["train", "val"]:
    (DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# Shuffle and split
random.seed(42)
image_stems = [img.stem.replace("annotated_", "") for img in images]
random.shuffle(image_stems)

split_idx   = max(1, int(len(image_stems) * 0.8))
train_stems = image_stems[:split_idx]
val_stems   = image_stems[split_idx:]

print(f"Train ({len(train_stems)}): {train_stems}")
print(f"Val   ({len(val_stems)}):   {val_stems}")

def copy_split(stems, split):
    for stem in stems:
        for prefix in ["annotated_", ""]:
            for ext in [".jpg", ".png"]:
                img_src = SRC_IMAGES / f"{prefix}{stem}{ext}"
                if img_src.exists():
                    shutil.copy(img_src, DATASET_DIR / "images" / split / f"{stem}.jpg")
                    break
        lbl_src = SRC_LABELS / f"{stem}.txt"
        if lbl_src.exists():
            shutil.copy(lbl_src, DATASET_DIR / "labels" / split / f"{stem}.txt")
        else:
            print(f"  ⚠️  Label not found: {stem}")

copy_split(train_stems, "train")
copy_split(val_stems,   "val")

train_count = len(list((DATASET_DIR / "images" / "train").glob("*")))
val_count   = len(list((DATASET_DIR / "images" / "val").glob("*")))
print(f"\n✅ Copied → train: {train_count} | val: {val_count}")

## 5. Create dataset.yaml

In [ ]:
dataset_yaml = {
    "path" : str(DATASET_DIR),
    "train": "images/train",
    "val"  : "images/val",
    "nc"   : 1,
    "names": {0: "grass_plant"}
}

yaml_path = DATASET_DIR / "dataset.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print("dataset.yaml:")
print(open(yaml_path).read())

## 6. Visualize Training Samples

In [ ]:
train_img_paths = list((DATASET_DIR / "images" / "train").glob("*"))[:4]
n = len(train_img_paths)

fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
if n == 1:
    axes = [axes]

for ax, img_path in zip(axes, train_img_paths):
    img = Image.open(img_path).convert("RGB")
    w, h = img.size
    ax.imshow(img)

    lbl_path = DATASET_DIR / "labels" / "train" / f"{img_path.stem}.txt"
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    _, cx, cy, bw, bh = map(float, parts)
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    rect = patches.Rectangle(
                        (x1, y1), bw*w, bh*h,
                        linewidth=2, edgecolor="lime", facecolor="none"
                    )
                    ax.add_patch(rect)

    ax.set_title(img_path.name, fontsize=9)
    ax.axis("off")

plt.suptitle("Training Samples with YOLO Labels", fontweight="bold")
plt.tight_layout()
plt.show()

## 7. Train YOLO26s 🚀

> **YOLO26s** — Ultralytics 2025 model | mAP 48.6 | 9.5M params | 2.5ms T4

In [ ]:
# Load YOLO26s pretrained on COCO (auto-downloads ~20MB)
model = YOLO("yolo26s.pt")

print("Model info:")
model.info()

# ── TRAINING ──────────────────────────────────────────────────────────────────
results = model.train(
    data         = str(yaml_path),
    epochs       = 100,           # increase to 200 for better results
    imgsz        = 640,
    batch        = 8,             # reduce to 4 if out-of-memory
    device       = 0,
    project      = "/kaggle/working/runs",
    name         = "grass_yolo26s",
    exist_ok     = True,

    # Optimizer
    optimizer    = "AdamW",
    lr0          = 0.001,
    lrf          = 0.01,
    weight_decay = 0.0005,

    # Augmentation — critical for small 10-image dataset
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    degrees      = 10.0,
    translate    = 0.1,
    scale        = 0.5,
    flipud       = 0.3,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.1,
    copy_paste   = 0.1,

    # Settings
    plots        = True,
    save         = True,
    verbose      = True,
    patience     = 30,           # early stopping after 30 epochs no improvement
)

print("\n✅ Training complete!")

## 8. Evaluate on Validation Set

In [ ]:
best_model_path = "/kaggle/working/runs/grass_yolo26s/weights/best.pt"
trained_model = YOLO(best_model_path)

metrics = trained_model.val(data=str(yaml_path))

print("\n── Validation Metrics ──────────────────")
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")

## 9. Inference on Val Images

In [ ]:
val_img_paths = list((DATASET_DIR / "images" / "val").glob("*"))

preds = trained_model.predict(
    source   = val_img_paths,
    conf     = 0.25,
    iou      = 0.45,
    save     = True,
    project  = "/kaggle/working/runs",
    name     = "grass_predict",
    exist_ok = True
)

fig, axes = plt.subplots(1, len(preds), figsize=(6 * len(preds), 5))
if len(preds) == 1:
    axes = [axes]

for ax, result in zip(axes, preds):
    img_rgb = result.plot()[:, :, ::-1]
    ax.imshow(img_rgb)
    ax.set_title(f"{Path(result.path).name}\n{len(result.boxes)} detections")
    ax.axis("off")

plt.suptitle("YOLO26s Predictions", fontweight="bold")
plt.tight_layout()
plt.savefig("/kaggle/working/val_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Training Curves

In [ ]:
results_png = Path("/kaggle/working/runs/grass_yolo26s/results.png")
if results_png.exists():
    img = Image.open(results_png)
    plt.figure(figsize=(16, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("YOLO26s Training Curves", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("results.png not found yet.")

## 11. Export to ONNX

In [ ]:
trained_model.export(format="onnx", imgsz=640, dynamic=False)
print("✅ Exported to ONNX")

# Uncomment for other formats:
# trained_model.export(format="tflite")      # Android / Edge
# trained_model.export(format="engine")      # TensorRT (fastest on Nvidia)

## 12. Download Trained Model

In [ ]:
import shutil
from IPython.display import FileLink, display

zip_path = "/kaggle/working/grass_yolo26s_trained"
shutil.make_archive(zip_path, "zip", "/kaggle/working/runs/grass_yolo26s")

size_mb = Path(zip_path + ".zip").stat().st_size / (1024 * 1024)
print(f"✅ Zipped → grass_yolo26s_trained.zip  ({size_mb:.2f} MB)")

display(FileLink("grass_yolo26s_trained.zip", result_html_prefix="Download: "))